# Modèle Multi-Tâche V5 - MobileNetV3Small

**Objectifs:**

- Backbone **MobileNetV3Small** (optimisé mobile) au lieu d'EfficientNetB0
- Dataset équilibré (même nombre d'images par classe)
- 4 classes d'ethnicité (sans "Autre"): Blanc, Noir, Asiatique, Indien
- Fine-tuning en 2 phases
- Modèle TFLite plus léger et plus rapide

**Avantages de MobileNetV3Small:**

- ~2.5M paramètres (vs ~5.3M pour EfficientNetB0)
- 2-3x plus rapide sur mobile
- TFLite ~6-10MB (vs ~15-20MB)


In [1]:
import os, sys, pathlib
import random
ROOT = pathlib.Path.cwd()
if not (ROOT / 'config.py').exists():
    for parent in ROOT.parents:
        if (parent / 'config.py').exists():
            ROOT = parent
            break
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from config import (
    BASE_DIR, DATA_DIR, RAW_DIR, IMAGES_DIR, ARTIFACTS_DIR,
    NOTEBOOKS_DIR, ANDROID_ASSETS_DIR, TESTS_DIR, DIAGNOSTICS_DIR,
    ensure_kaggle_credentials, ensure_kaggle_dataset, describe_config,
    artifacts_for,
 )

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks, regularizers
from tensorflow.keras.applications import EfficientNetB0, MobileNetV3Large, MobileNetV3Small
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.regularizers import l2
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import json
import time
from PIL import Image
from pathlib import Path
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

print('Working dir set to', ROOT)
print(describe_config())


2026-03-27 22:38:29.360531: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-27 22:38:29.362188: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-27 22:38:29.383451: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-27 22:38:29.383488: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-27 22:38:29.384373: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

TensorFlow version: 2.15.0
GPU disponible: []
Working dir set to /home/carld/face-analysis-deep-learning
{'BASE_DIR': '/home/carld/face-analysis-deep-learning', 'DATA_DIR': '/home/carld/face-analysis-deep-learning/data', 'RAW_DIR': '/home/carld/face-analysis-deep-learning/data/raw', 'IMAGES_DIR': '/home/carld/face-analysis-deep-learning/data/UTKFace', 'ARTIFACTS_DIR': '/home/carld/face-analysis-deep-learning/artifacts', 'ANDROID_ASSETS_DIR': '/home/carld/face-analysis-deep-learning/FacePredictor/app/src/main/assets', 'TESTS_DIR': '/home/carld/face-analysis-deep-learning/tests', 'DIAGNOSTICS_DIR': '/home/carld/face-analysis-deep-learning/diagnostics', 'DATASET_ID': 'jangedoo/utkface-new'}


2026-03-27 22:38:32.242327: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-03-27 22:38:32.242616: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## Configuration


In [2]:
# Chemins
DATA_DIR = IMAGES_DIR
OUTPUT_DIR = artifacts_for("multitask", "multitask_model_v5_mobilenet")
ARTIFACTS_DIR = OUTPUT_DIR

# Hyperparamètres
IMG_SIZE = 224
BATCH_SIZE = 32
WARMUP_EPOCHS = 25
FINETUNE_EPOCHS = 50
INITIAL_LR = 1e-3
FINETUNE_LR = 1e-5

# Seed global pour reproductibilité
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Splits
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

# Classes - SANS "Autre" (4 classes seulement)
ETHNICITY_CLASSES = ["Blanc", "Noir", "Asiatique", "Indien"]
GENDER_CLASSES = ["Homme", "Femme"]

print(f"Dataset: {DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Configuration V5 (MobileNetV3Small):")
print(f"  - Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Backbone: MobileNetV3Small")
print(f"  - Ethnicités: {ETHNICITY_CLASSES}")

Dataset: /home/carld/face-analysis-deep-learning/data/UTKFace
Output: /home/carld/face-analysis-deep-learning/artifacts/multitask/multitask_model_v5_mobilenet
Configuration V5 (MobileNetV3Small):
  - Image size: 224x224
  - Batch size: 32
  - Backbone: MobileNetV3Small
  - Ethnicités: ['Blanc', 'Noir', 'Asiatique', 'Indien']


## Chargement et Équilibrage du Dataset


In [3]:
def parse_utkface_filename(filepath):
    """Parse le nom de fichier UTKFace: [age]_[gender]_[race]_[date&time].jpg"""
    filename = Path(filepath).stem
    parts = filename.split("_")

    if len(parts) >= 3:
        try:
            age = int(parts[0])
            gender = int(parts[1])  # 0: Male, 1: Female
            ethnicity = int(parts[2])  # 0-4

            # Filtrer: on ne garde que les ethnicités 0-3 (pas "Others" qui est 4)
            if 0 <= age <= 116 and gender in [0, 1] and 0 <= ethnicity <= 3:
                return {
                    "filepath": str(filepath),
                    "age": age,
                    "gender": gender,
                    "ethnicity": ethnicity,
                }
        except (ValueError, IndexError):
            pass
    return None


# Charger toutes les images
image_paths = list(DATA_DIR.glob("*.jpg")) + list(DATA_DIR.glob("*.JPG"))
print(f"Images trouvées: {len(image_paths)}")

records = []
for path in tqdm(image_paths, desc="Parsing filenames"):
    record = parse_utkface_filename(path)
    if record:
        records.append(record)

df = pd.DataFrame(records)
print(f"\nImages valides (sans 'Autre'): {len(df)}")

# Distribution
print(f"\nDistribution des genres:")
print(df["gender"].value_counts())
print(f"\nDistribution des ethnicités:")
for i, name in enumerate(ETHNICITY_CLASSES):
    count = len(df[df["ethnicity"] == i])
    print(f"  {i}: {name} = {count}")

Images trouvées: 23708


Parsing filenames: 100%|██████████| 23708/23708 [00:00<00:00, 335084.12it/s]


Images valides (sans 'Autre'): 22013

Distribution des genres:
gender
0    11631
1    10382
Name: count, dtype: int64

Distribution des ethnicités:
  0: Blanc = 10078
  1: Noir = 4526
  2: Asiatique = 3434
  3: Indien = 3975


In [4]:
def balance_dataset(df, seed=42):
    """Équilibre le dataset pour avoir le même nombre d'images par groupe (genre, ethnicité)"""
    np.random.seed(seed)

    groups = df.groupby(["gender", "ethnicity"])
    group_counts = groups.size()
    print("Nombre d'images par groupe (avant équilibrage):")
    for (g, e), count in group_counts.items():
        print(f"  {GENDER_CLASSES[g]}, {ETHNICITY_CLASSES[e]}: {count}")

    min_count = group_counts.min()
    print(f"\nMinimum: {min_count} images par groupe")

    balanced_dfs = []
    for (gender, ethnicity), group_df in groups:
        sampled = group_df.sample(n=min_count, random_state=seed)
        balanced_dfs.append(sampled)

    df_balanced = pd.concat(balanced_dfs, ignore_index=True)
    df_balanced = df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)

    return df_balanced, min_count


df_balanced, samples_per_group = balance_dataset(df)

print(f"\nDataset équilibré: {len(df_balanced)} images")
print(f"  - {samples_per_group} images par groupe (genre, ethnicité)")

Nombre d'images par groupe (avant équilibrage):
  Homme, Blanc: 5477
  Homme, Noir: 2318
  Homme, Asiatique: 1575
  Homme, Indien: 2261
  Femme, Blanc: 4601
  Femme, Noir: 2208
  Femme, Asiatique: 1859
  Femme, Indien: 1714

Minimum: 1575 images par groupe

Dataset équilibré: 12600 images
  - 1575 images par groupe (genre, ethnicité)


## Split Train/Val/Test


In [5]:
df_balanced["stratify_col"] = (
    df_balanced["gender"].astype(str) + "_" + df_balanced["ethnicity"].astype(str)
)

X = df_balanced["filepath"].values
y_age = df_balanced["age"].values.astype(np.float32)
y_gender = df_balanced["gender"].values.astype(np.int32)
y_ethnicity = df_balanced["ethnicity"].values.astype(np.int32)
stratify_col = df_balanced["stratify_col"].values

# Premier split: train+val vs test
(
    X_trainval,
    X_test,
    y_age_trainval,
    y_age_test,
    y_gender_trainval,
    y_gender_test,
    y_eth_trainval,
    y_eth_test,
) = train_test_split(
    X,
    y_age,
    y_gender,
    y_ethnicity,
    test_size=TEST_SPLIT,
    random_state=SEED,
    stratify=stratify_col,
)

stratify_trainval = [f"{g}_{e}" for g, e in zip(y_gender_trainval, y_eth_trainval)]

# Deuxième split: train vs val
val_ratio = VAL_SPLIT / (1 - TEST_SPLIT)
(
    X_train,
    X_val,
    y_age_train,
    y_age_val,
    y_gender_train,
    y_gender_val,
    y_eth_train,
    y_eth_val,
) = train_test_split(
    X_trainval,
    y_age_trainval,
    y_gender_trainval,
    y_eth_trainval,
    test_size=val_ratio,
    random_state=SEED,
    stratify=stratify_trainval,
)

print(f"Split des données:")
print(f"  - Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"  - Val: {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"  - Test: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

Split des données:
  - Train: 8819 (70.0%)
  - Val: 1891 (15.0%)
  - Test: 1890 (15.0%)


## Data Augmentation


In [6]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.15),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomBrightness(0.2),
        layers.RandomContrast(0.2),
    ],
    name="data_augmentation",
)

print("Data augmentation configurée")

Data augmentation configurée


## Pipeline de Données


In [7]:
def load_and_preprocess(filepath, age, gender, ethnicity):
    """Charge et prétraite une image."""
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)  # [0, 255]
    return img, {
        "age_output": age,
        "gender_output": gender,
        "ethnicity_output": ethnicity,
    }


def create_dataset(filepaths, ages, genders, ethnicities, training=False):
    """Crée un tf.data.Dataset."""
    ds = tf.data.Dataset.from_tensor_slices((filepaths, ages, genders, ethnicities))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.shuffle(buffer_size=2000)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = create_dataset(
    X_train, y_age_train, y_gender_train, y_eth_train, training=True
)
val_ds = create_dataset(X_val, y_age_val, y_gender_val, y_eth_val, training=False)
test_ds = create_dataset(X_test, y_age_test, y_gender_test, y_eth_test, training=False)

print(f"Datasets créés")

Datasets créés


## Architecture du Modèle - MobileNetV3Small

**Changement principal:** Utilisation de `MobileNetV3Small` au lieu de `EfficientNetB0`

MobileNetV3Small a ~158 couches (vs ~238 pour EfficientNetB0), donc on ajuste le fine-tuning.


In [8]:
def build_multitask_model_v5(input_shape=(IMG_SIZE, IMG_SIZE, 3), training=True):
    """
    Modèle multi-tâche V5 avec MobileNetV3Small.

    Avantages vs EfficientNetB0:
    - 2x moins de paramètres
    - 2-3x plus rapide sur mobile
    - TFLite plus petit
    """

    inputs = layers.Input(shape=input_shape, name="input_image")

    # Data augmentation (seulement en training)
    x = data_augmentation(inputs) if training else inputs

    # ════════════════════════════════════════════════════════════
    # BACKBONE: MobileNetV3Small (au lieu de EfficientNetB0)
    # ════════════════════════════════════════════════════════════
    backbone = MobileNetV3Small(
        include_top=False,
        weights="imagenet",
        input_tensor=x,
        include_preprocessing=True,  # Normalisation intégrée
    )
    backbone.trainable = False  # Gelé pour le warmup

    features = backbone.output

    # Double pooling pour plus de features
    gap = layers.GlobalAveragePooling2D(name="gap")(features)
    gmp = layers.GlobalMaxPooling2D(name="gmp")(features)
    pooled = layers.Concatenate(name="concat_pooling")([gap, gmp])

    # Couches partagées
    shared = layers.Dense(
        512, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="shared_dense1"
    )(pooled)
    shared = layers.BatchNormalization(name="shared_bn1")(shared)
    shared = layers.Dropout(0.4, name="shared_dropout1")(shared)

    shared = layers.Dense(
        256, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="shared_dense2"
    )(shared)
    shared = layers.BatchNormalization(name="shared_bn2")(shared)
    shared = layers.Dropout(0.3, name="shared_dropout2")(shared)

    # === HEAD AGE (Régression) ===
    age_branch = layers.Dense(
        128, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="age_dense1"
    )(shared)
    age_branch = layers.BatchNormalization(name="age_bn")(age_branch)
    age_branch = layers.Dropout(0.3, name="age_dropout")(age_branch)
    age_output = layers.Dense(1, activation="linear", name="age_output")(age_branch)

    # === HEAD GENRE (Classification binaire) ===
    gender_branch = layers.Dense(
        128, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="gender_dense1"
    )(shared)
    gender_branch = layers.BatchNormalization(name="gender_bn")(gender_branch)
    gender_branch = layers.Dropout(0.3, name="gender_dropout")(gender_branch)
    gender_output = layers.Dense(1, activation="sigmoid", name="gender_output")(
        gender_branch
    )

    # === HEAD ETHNICITÉ (4 classes) ===
    ethnicity_branch = layers.Dense(
        256, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="ethnicity_dense1"
    )(shared)
    ethnicity_branch = layers.BatchNormalization(name="ethnicity_bn1")(ethnicity_branch)
    ethnicity_branch = layers.Dropout(0.4, name="ethnicity_dropout1")(ethnicity_branch)
    ethnicity_branch = layers.Dense(
        128, activation="relu", kernel_regularizer=regularizers.l2(1e-4), name="ethnicity_dense2"
    )(ethnicity_branch)
    ethnicity_branch = layers.BatchNormalization(name="ethnicity_bn2")(ethnicity_branch)
    ethnicity_branch = layers.Dropout(0.3, name="ethnicity_dropout2")(ethnicity_branch)
    ethnicity_output = layers.Dense(4, activation="softmax", name="ethnicity_output")(
        ethnicity_branch
    )

    model = Model(
        inputs=inputs,
        outputs=[age_output, gender_output, ethnicity_output],
        name="multitask_face_model_v5_mobilenet",
    )

    return model, backbone


model, backbone = build_multitask_model_v5(training=True)

print(f"\n{'='*60}")
print(f"MODÈLE V5 - MobileNetV3Small")
print(f"{'='*60}")
print(f"Backbone: MobileNetV3Small ({len(backbone.layers)} couches)")
print(f"Total params: {model.count_params():,}")
model.summary()


MODÈLE V5 - MobileNetV3Small
Backbone: MobileNetV3Small (230 couches)
Total params: 1,831,670
Model: "multitask_face_model_v5_mobilenet"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_image (InputLayer)    [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 data_augmentation (Sequent  (None, 224, 224, 3)          0         ['input_image[0][0]']         
 ial)                                                                                             
                                                                                                  
 rescaling (Rescaling)       (None, 224, 224, 3)          0         ['data_augmentation[0][0]']   
                                                                      

## Compilation


In [9]:
steps_per_epoch = len(X_train) // BATCH_SIZE

warmup_lr_schedule = CosineDecay(
    initial_learning_rate=INITIAL_LR,
    decay_steps=steps_per_epoch * WARMUP_EPOCHS,
    alpha=0.1,
)

losses = {
    "age_output": keras.losses.Huber(delta=5.0),
    "gender_output": keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    "ethnicity_output": keras.losses.SparseCategoricalCrossentropy(),
}

loss_weights = {
    "age_output": 0.5,
    "gender_output": 1.5,
    "ethnicity_output": 1.5,
}

metrics = {
    "age_output": ["mae"],
    "gender_output": ["accuracy"],
    "ethnicity_output": ["accuracy"],
}

model.compile(
    optimizer=AdamW(learning_rate=warmup_lr_schedule, weight_decay=1e-5),
    loss=losses,
    loss_weights=loss_weights,
    metrics=metrics,
)

print("Modèle compilé!")

Modèle compilé!


## Phase 1: Warmup (Backbone gelé)


In [10]:
warmup_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=str(ARTIFACTS_DIR / "multitask_v5_mobilenet_warmup.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    ),
]

print("=" * 60)
print("PHASE 1: WARMUP (Backbone MobileNetV3Small gelé)")
print("=" * 60)

warmup_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=WARMUP_EPOCHS,
    callbacks=warmup_callbacks,
    verbose=1,
)

PHASE 1: WARMUP (Backbone MobileNetV3Small gelé)
Epoch 1/25
  9/276 [..............................] - ETA: 28s - loss: 81.5968 - age_output_loss: 154.0459 - gender_output_loss: 0.9605 - ethnicity_output_loss: 1.9677 - age_output_mae: 33.2276 - gender_output_accuracy: 0.5000 - ethnicity_output_accuracy: 0.2465

KeyboardInterrupt: 

## Phase 2: Fine-tuning (Backbone partiellement dégelé)

MobileNetV3Small a ~158 couches. On dégèle les 40 dernières couches.


In [ ]:
print("=" * 60)
print("PHASE 2: FINE-TUNING (Backbone partiellement dégelé)")
print("=" * 60)

# Dégeler les dernières couches du backbone
backbone.trainable = True

# MobileNetV3Small a ~158 couches, on en dégèle 40
freeze_until = len(backbone.layers) - 40

for layer in backbone.layers[:freeze_until]:
    layer.trainable = False

trainable_count = sum([1 for l in backbone.layers if l.trainable])
print(f"Couches backbone: {len(backbone.layers)}")
print(f"Couches entraînables: {trainable_count}")

# Recompiler avec learning rate plus faible
finetune_lr_schedule = CosineDecay(
    initial_learning_rate=FINETUNE_LR,
    decay_steps=steps_per_epoch * FINETUNE_EPOCHS,
    alpha=0.01,
)

model.compile(
    optimizer=AdamW(learning_rate=finetune_lr_schedule, weight_decay=1e-5),
    loss=losses,
    loss_weights=loss_weights,
    metrics=metrics,
)

finetune_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=str(ARTIFACTS_DIR / "multitask_v5_mobilenet_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        verbose=1,
    ),
]

finetune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    callbacks=finetune_callbacks,
    verbose=1,
)

## Évaluation


In [ ]:
# Charger le meilleur modèle
best_model = keras.models.load_model(
    ARTIFACTS_DIR / "multitask_v5_mobilenet_best.keras"
)

# Prédictions
predictions = best_model.predict(test_ds, verbose=1)
y_pred_age = predictions[0].flatten()
y_pred_gender = (predictions[1].flatten() > 0.5).astype(int)
y_pred_ethnicity = np.argmax(predictions[2], axis=1)

# Métriques
mae_age = np.mean(np.abs(y_age_test - y_pred_age))
rmse_age = np.sqrt(np.mean((y_age_test - y_pred_age) ** 2))
acc_gender = accuracy_score(y_gender_test, y_pred_gender)
f1_gender = f1_score(y_gender_test, y_pred_gender)
acc_ethnicity = accuracy_score(y_eth_test, y_pred_ethnicity)
f1_ethnicity_macro = f1_score(y_eth_test, y_pred_ethnicity, average="macro")

print("\n" + "=" * 60)
print("RÉSULTATS V5 (MobileNetV3Small) SUR LE TEST SET")
print("=" * 60)
print(f"\nAGE:")
print(f"   MAE: {mae_age:.2f} années")
print(f"   RMSE: {rmse_age:.2f} années")
print(f"\nGENRE:")
print(f"   Accuracy: {acc_gender*100:.2f}%")
print(f"   F1-Score: {f1_gender:.4f}")
print(f"\nETHNICITÉ:")
print(f"   Accuracy: {acc_ethnicity*100:.2f}%")
print(f"   F1-Score (macro): {f1_ethnicity_macro:.4f}")

In [ ]:
print("\n" + "=" * 60)
print("RAPPORT DE CLASSIFICATION - GENRE")
print("=" * 60)
print(classification_report(y_gender_test, y_pred_gender, target_names=GENDER_CLASSES))

print("\n" + "=" * 60)
print("RAPPORT DE CLASSIFICATION - ETHNICITÉ")
print("=" * 60)
print(
    classification_report(y_eth_test, y_pred_ethnicity, target_names=ETHNICITY_CLASSES)
)

## Matrices de Confusion


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Genre
cm_gender = confusion_matrix(y_gender_test, y_pred_gender)
sns.heatmap(
    cm_gender,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=GENDER_CLASSES,
    yticklabels=GENDER_CLASSES,
    ax=axes[0],
)
axes[0].set_xlabel("Prédit")
axes[0].set_ylabel("Réel")
axes[0].set_title(f"Genre - Accuracy: {acc_gender*100:.2f}%")

# Ethnicité
cm_ethnicity = confusion_matrix(y_eth_test, y_pred_ethnicity)
sns.heatmap(
    cm_ethnicity,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=ETHNICITY_CLASSES,
    yticklabels=ETHNICITY_CLASSES,
    ax=axes[1],
)
axes[1].set_xlabel("Prédit")
axes[1].set_ylabel("Réel")
axes[1].set_title(f"Ethnicité - Accuracy: {acc_ethnicity*100:.2f}%")

plt.tight_layout()
plt.savefig(
    ARTIFACTS_DIR / "multitask_v5_mobilenet_confusion.png", dpi=150, bbox_inches="tight"
)
plt.show()

## Sauvegarde et Conversion TFLite


In [ ]:
# Sauvegarder le modèle final
final_model_path = ARTIFACTS_DIR / "multitask_v5_mobilenet_final.keras"
best_model.save(final_model_path)
print(f"Modèle Keras sauvegardé: {final_model_path}")

In [ ]:
# Conversion TFLite avec optimisations
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]  # Quantization float16

tflite_model = converter.convert()

tflite_path = ARTIFACTS_DIR / "multitask_model_v5_mobilenet.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

tflite_size_mb = tflite_path.stat().st_size / (1024 * 1024)
print(f"Modèle TFLite sauvegardé: {tflite_path}")
print(f"Taille: {tflite_size_mb:.2f} MB")

## Métadonnées


In [ ]:
import json

model_info = {
    "model_type": "multitask_v5_mobilenet",
    "version": "5.0",
    "backbone": "MobileNetV3Small",
    "img_size": IMG_SIZE,
    "input_range": [0, 255],
    "dataset": {
        "total_images": len(df_balanced),
        "images_per_group": int(samples_per_group),
        "balanced": True,
    },
    "outputs": {
        "age": {
            "type": "regression",
            "output_name": "age_output",
            "description": "Age en années",
        },
        "gender": {
            "type": "binary_classification",
            "output_name": "gender_output",
            "classes": ["Homme", "Femme"],
            "threshold": 0.5,
        },
        "ethnicity": {
            "type": "multiclass_classification",
            "output_name": "ethnicity_output",
            "num_classes": 4,
            "classes": ["Blanc", "Noir", "Asiatique", "Indien"],
        },
    },
    "metrics": {
        "age_mae": float(mae_age),
        "age_rmse": float(rmse_age),
        "gender_accuracy": float(acc_gender),
        "gender_f1": float(f1_gender),
        "ethnicity_accuracy": float(acc_ethnicity),
        "ethnicity_f1_macro": float(f1_ethnicity_macro),
    },
    "tflite_size_mb": float(tflite_size_mb),
}

info_path = ARTIFACTS_DIR / "multitask_v5_mobilenet_info.json"
with open(info_path, "w") as f:
    json.dump(model_info, f, indent=2)

print(f"Métadonnées sauvegardées: {info_path}")

## Résumé Final


In [ ]:
print("=" * 60)
print("RÉSUMÉ DU MODÈLE MULTI-TÂCHE V5 (MobileNetV3Small)")
print("=" * 60)
print(f"\nBackbone: MobileNetV3Small (optimisé mobile)")
print(f"Input: {IMG_SIZE}x{IMG_SIZE}x3")
print(f"\nDataset:")
print(f"  - {len(df_balanced)} images équilibrées")
print(f"  - 4 classes d'ethnicité (sans 'Autre')")
print(f"\nPerformances sur le test set:")
print(f"  AGE:")
print(f"     - MAE: {mae_age:.2f} années")
print(f"     - RMSE: {rmse_age:.2f} années")
print(f"  GENRE:")
print(f"     - Accuracy: {acc_gender*100:.2f}%")
print(f"     - F1-Score: {f1_gender:.4f}")
print(f"  ETHNICITÉ:")
print(f"     - Accuracy: {acc_ethnicity*100:.2f}%")
print(f"     - F1-Score (macro): {f1_ethnicity_macro:.4f}")
print(f"\nTaille TFLite: {tflite_size_mb:.2f} MB")
print(f"\nFichiers sauvegardés:")
print(f"  - {final_model_path}")
print(f"  - {tflite_path}")
print(f"  - {info_path}")
print("=" * 60)